In [1]:
# Select the mode and configure the feature-build inputs and artifacts.

import os

mode = "replay"
os.environ["COMMENTGAP_MODE"] = mode
os.environ['COMMENTGAP_FROZEN_REPLAY'] = '1' if mode == 'recompute' else '0'

from pathlib import Path
import pandas as pd
from IPython.display import display
import json
from commentgap_analysis.public_replay import verify_public_stage
if mode != 'replay':
    import pyarrow.parquet as pq
    from commentgap_analysis.paths import ExecutionContext
    from commentgap_analysis.frozen_inputs import unavailable
    from commentgap_analysis.notebook_paths import configure_notebook_stage

if mode == 'replay':
    PUBLIC_ROOT = Path(os.environ.get('COMMENTGAP_PUBLIC_ARTIFACT_ROOT', Path.cwd() / 'artifacts/canonical')).resolve()
    verify_public_stage('02', root=PUBLIC_ROOT)
elif mode == 'recompute':
    _cg_context = ExecutionContext.from_values(mode="frozen")
    os.environ['COMMENTGAP_DATA_ROOT'] = str(_cg_context.read_root('raw_scrape'))
    os.environ['COMMENTGAP_MODEL_ROOT'] = str(_cg_context.read_root('model_data'))
    os.environ['COMMENTGAP_EMBEDDING_ROOT'] = str(_cg_context.read_root('embeddings'))
    os.environ['COMMENTGAP_SIMILARITY_ROOT'] = str(_cg_context.read_root('similarities'))
    os.environ['COMMENTGAP_AQUA_ROOT'] = str(_cg_context.read_root('aqua'))
    os.environ['COMMENTGAP_FEATURE_ROOT'] = str(_cg_context.read_root('features'))
elif mode in ('fresh', 'resume'):
    if mode == 'fresh':
        os.environ.setdefault('COMMENTGAP_RUN_LONG_FEATURE_ASSEMBLY', '1')
    configure_notebook_stage('02')


# 2. Build the production 2025 model-feature store

This notebook is the auditable orchestration record for the complete feature pipeline. The production run has achieved the original objective: it created validated, resumable feature-family stores and the matched all/root-comment choice sets consumed by stages 3–8. Long transformer jobs are launched as shell commands on the Linux A5000 host; ordinary notebook execution inspects their manifests and validates the completed outputs without recomputing them.

### Execution modes for this notebook

`replay` (default) verifies and displays the compact public aggregate products from `artifacts/canonical/`. `recompute` executes the saved-product cells using private `data/` and `artifacts/archive/recompute/`. `fresh` runs this stage’s producer from retained inputs; see [reproduction modes](docs/reproduction.md) for prerequisites. `resume` continues a recorded producer run.


In [2]:
# Inspect scrape QA and set the source paths used by feature assembly.

if mode != 'replay':


    DATA_ROOT = Path(os.environ["COMMENTGAP_DATA_ROOT"])
    MODEL_ROOT = Path(os.environ["COMMENTGAP_MODEL_ROOT"])
    EMBEDDING_ROOT = Path(os.environ["COMMENTGAP_EMBEDDING_ROOT"])
    SIMILARITY_ROOT = Path(os.environ["COMMENTGAP_SIMILARITY_ROOT"])
    AQUA_ROOT = Path(os.environ["COMMENTGAP_AQUA_ROOT"])
    FEATURE_ROOT = Path(os.environ["COMMENTGAP_FEATURE_ROOT"])

    qa = json.loads((DATA_ROOT / "qa_summary/year=2025/summary.json").read_text())
    assert qa["passed"] and qa["nonterminal_stories"] == 0
    {
        "articles": qa.get("article_rows"),
        "comments": qa.get("comment_rows"),
        "status_counts": qa.get("status_counts"),
        "sticky_descendant_rows": qa.get("sticky_descendant_rows"),
    }


## Production dependency graph

1. **BGE-M3 embeddings:** normalized comment and article-passage vectors for December 2024 and 2025.
2. **Semantic scalars:** article similarity plus strict-prior all-comment and root-comment novelty for 2025.
3. **AQuA:** twenty German deliberative-quality dimensions in the isolated Python 3.10 runtime; their continuous expected ordinal scores enter the primary models. Hard labels and composites remain descriptive/sensitivity features.
4. **Main feature assembly:** discussion and author history, deterministic local-text measures, CardiffNLP XLM-T sentiment, TextDetox toxicity, semantic joins, AQuA joins, length adjustments, and matched root/all choice sets.

Each expensive family has its own checkpoint identity. Repeating a compatible command resumes or reuses completed story shards; do not add `--overwrite` to a routine restart.

## 1. BGE-M3 embedding store — long CUDA run

The production run omitted `--year`, deliberately allowing discovery of both December 2024 and all 2025 partitions. It began at batch 64 and adaptively increased to 256; vectors were stored as float16 while inference remained float32.

```bash
source .venv/bin/activate
python -m pip install -e .

commentgap-embed \
  --model-id BAAI/bge-m3 \
  --revision 5617a9f61b028005a4858fdac845db406aefb181 \
  --device cuda \
  --batch-size 64 \
  --max-batch-size 256 \
  --min-batch-size 4 \
  --storage-dtype float16 \
  --progress-every-stories 25
```

The completed manifest recorded 48,061 articles, 9,941,495 eligible comments, a 1,024-dimensional embedding, and the `COMPLETE_SOURCE` watermark.

## 2. Precomputed semantic features — long CPU/RAM run

This stage reads the completed embedding store and does not reload BGE-M3. The 2025 model requires article similarity, novelty from earlier comments, and novelty from earlier roots.

```bash
commentgap-similarity \
  --model-id BAAI/bge-m3 \
  --revision 5617a9f61b028005a4858fdac845db406aefb181 \
  --year 2025 \
  --exact-novelty-threshold 5000 \
  --progress-every-stories 100
```

The production similarity build signature is resolved from its manifest; the feature assembler rejects a source-fingerprint mismatch.

## 3. AQuA deliberative-quality store — long isolated CUDA run

The numerical production source was run with the following tuned A5000 settings: batch ceiling 128, padded-token budget 2,048, and cross-story windows of at most 500 stories or 200,000 rows. The historical run accidentally included `--allow-unverified-parity`, so it was correctly watermarked as a pilot and subsequently promoted after upstream parity verification. **For a new production run with verified parity, omit that flag.**

```bash
# Historical numerical run (retained here for exact reproducibility).
commentgap-aqua \
  --runtime-python .venv-aqua/bin/python \
  --device cuda \
  --batch-size 128 \
  --max-batch-tokens 2048 \
  --window-max-stories 500 \
  --window-max-rows 200000 \
  --progress-every-stories 100 \
  --allow-unverified-parity

# Audited metadata promotion after commentgap-aqua-parity verified the fixture.
commentgap-aqua-promote \
  --source-store "${COMMENTGAP_AQUA_ROOT:-data/derived/aqua}" \
  --output-root "${COMMENTGAP_AQUA_ROOT:-data/derived/aqua}" \
  --data-root "${COMMENTGAP_DATA_ROOT:-data/raw/scrape_2025}" \
  --year 2025 \
  --artifact-manifest aqua_runtime/artifacts.json \
  --progress-every-stories 25
```

The original promotion audit note was local and is not tracked; this is recorded as a reproducibility gap in `papers/paper1/README.md`. Promotion validated all 9,222,852 rows across 44,022 stories and changed only signatures/watermarks; numeric predictions were not recomputed.

## 4. Main feature assembly — long CUDA plus CPU/RAM run

This is the stage that fulfills the original purpose of 06A. It resolves and validates the similarity and production AQuA stores; reuses or creates local-text, CardiffNLP sentiment, and TextDetox toxicity story shards; computes strict-prior discussion/author history with the December 2024 lookback; and writes the final choice sets. The default immutable revisions are CardiffNLP `f2f1202b1bdeb07342385c3f807f9c07cd8f5cf8` and TextDetox `cde9d07ac4df2af9c02d2461dee068bf04a58728`.

Production command:

```bash
commentgap-features \
  --data-root "${COMMENTGAP_DATA_ROOT:-data/raw/scrape_2025}" \
  --output-root "${COMMENTGAP_FEATURE_ROOT:-data/derived/features}" \
  --similarity-root "${COMMENTGAP_SIMILARITY_ROOT:-data/derived/similarities}" \
  --aqua-store "${COMMENTGAP_AQUA_ROOT:-data/derived/aqua}" \
  --lookback-root "${COMMENTGAP_DATA_ROOT:-data/raw/scrape_2025}" \
  --year 2025 \
  --device cuda \
  --sentiment-batch-size 32 \
  --toxicity-batch-size 16 \
  --progress-every-stories 100
```

`--lookback-root data/scrape_2025` points at the same multi-year collection root and supplies its `year=2024` partition. Do not use `--overwrite` when resuming.

In [3]:
# Load the public feature inventory or run the feature-building pipeline.

if mode == 'replay':
    feature_inventory = json.loads((PUBLIC_ROOT / 'shared/features/feature_inventory_public.json').read_text())
    feature_columns = feature_inventory['column_names']
    feature_contract = pd.DataFrame([{'rows': feature_inventory['row_count'], 'columns': len(feature_columns),
        'aqua expected features': sum(n.startswith('aqua_') and n.endswith('_expected') for n in feature_columns)}])
elif mode == 'recompute':
    required_feature_products = [FEATURE_ROOT / name for name in (
        "choice_set_root.parquet", "choice_set_all.parquet",
        "feature_manifest.json", "provenance_manifest.json",
    )]
    missing_feature_products = [path for path in required_feature_products if not path.is_file()]
    if missing_feature_products:
        unavailable("saved feature assembly", missing_feature_products[0], "Complete feature products are required for notebooks 03–09.")
    else:
        display({"saved_feature_root": str(FEATURE_ROOT), "products": [path.name for path in required_feature_products], "assembly": "loaded from canonical products; no feature assembly run"})
elif mode in ('fresh', 'resume'):
    get_ipython().run_cell_magic("bash", "", "set -euo pipefail\nif [[ \"${COMMENTGAP_RUN_LONG_FEATURE_ASSEMBLY:-0}\" != \"1\" ]]; then echo \"Inspection mode: set COMMENTGAP_RUN_LONG_FEATURE_ASSEMBLY=1 before starting Jupyter to execute the production assembly.\"; exit 0; fi\ncommentgap-features --data-root \"${COMMENTGAP_DATA_ROOT:-data/raw/scrape_2025}\" --output-root \"${COMMENTGAP_FEATURE_ROOT:-data/derived/features}\" --similarity-root \"${COMMENTGAP_SIMILARITY_ROOT:-data/derived/similarities}\" --aqua-store \"${COMMENTGAP_AQUA_ROOT:-data/derived/aqua}\" --lookback-root \"${COMMENTGAP_DATA_ROOT:-data/raw/scrape_2025}\" --year 2025 --device cuda --sentiment-batch-size 32 --toxicity-batch-size 16 --progress-every-stories 100")


## Acceptance checks

These checks are deliberately manifest- and Parquet-metadata-based, so they do not load the 214 MB root or 721 MB all-comment choice set into pandas. The deeper DuckDB key/null/selection audit was run separately and passed.

In [4]:
# Compare the saved feature manifests with the public column contract.

if mode != 'replay':
    state = json.loads((FEATURE_ROOT / "build_state.json").read_text())
    provenance = json.loads((FEATURE_ROOT / "provenance_manifest.json").read_text())
    registry = json.loads((FEATURE_ROOT / "feature_manifest.json").read_text())

    assert state["status"] == "complete"
    assert provenance["watermark"] == "INFERENCE"
    assert provenance["models"]["aqua"]["watermark"] == "PRODUCTION"
    assert provenance["execution"]["choice_assembly_strategy"] == "bounded_cross_story_windows"

    # Check completion and provenance before trusting the saved feature products.
    expected = [
        FEATURE_ROOT / "choice_set_root.parquet",
        FEATURE_ROOT / "choice_set_all.parquet",
        FEATURE_ROOT / "tie_diagnostics_root.parquet",
        FEATURE_ROOT / "tie_diagnostics_all.parquet",
        FEATURE_ROOT / "feature_manifest.json",
        FEATURE_ROOT / "provenance_manifest.json",
        FEATURE_ROOT / "novelty_validation.json",
    ]
    missing = [str(path) for path in expected if not path.exists()]
    assert not missing, missing
    assert not list(FEATURE_ROOT.rglob("*.tmp"))

    for scope in ("root", "all"):
        rows = pq.ParquetFile(FEATURE_ROOT / f"choice_set_{scope}.parquet").metadata.num_rows
        tie_rows = pq.ParquetFile(FEATURE_ROOT / f"tie_diagnostics_{scope}.parquet").metadata.num_rows
        assert rows == provenance[scope]["candidate_rows"]
        assert tie_rows == provenance[scope]["eligible_stories"]
        model_features = registry["models"][scope]["features"]
        aqua_expected = [
            feature for feature in model_features
            if feature.startswith("aqua_") and feature.endswith("_expected")
        ]
        assert len(aqua_expected) == 20
        assert "aqua_score_expected" not in model_features

    {
        "build_signature": state["build_signature"],
        "dataset_fingerprint": provenance["source"]["dataset_fingerprint"],
        "aqua_build_signature": provenance["models"]["aqua"]["build_signature"],
        "root": provenance["root"],
        "all": provenance["all"],
        "root_model_features": len(registry["models"]["root"]["features"]),
        "all_model_features": len(registry["models"]["all"]["features"]),
    }

if mode != 'replay':

    # Use Parquet metadata to inspect the full table without loading its rows.
    feature_schema = pq.ParquetFile(FEATURE_ROOT / 'choice_set_all.parquet')
    feature_columns = feature_schema.schema_arrow.names
    feature_inventory = {'row_count': feature_schema.metadata.num_rows, 'column_names': feature_columns}

feature_contract = pd.DataFrame([{'rows': feature_inventory['row_count'], 'columns': len(feature_columns),
    'aqua expected features': sum(n.startswith('aqua_') and n.endswith('_expected') for n in feature_columns)}])
display(feature_contract)
display(pd.DataFrame({'column': feature_columns}))


,rows,columns,aqua expected features
0,2394962,105,21


,column
0,story_id
1,comment_id
2,article_year
3,article_month
4,candidate_scope
...,...
100,log_author_prior_comments_story
101,is_reply
102,log_depth
103,log_branch_prior_comments


## Downstream handoff

`07_stacked_selection_models.Rmd` and the stage-8 rankers both read `choice_set_root.parquet` and `choice_set_all.parquet` and obtain the primary feature lists from `feature_manifest.json`. The AQuA-backed contract contains 39 root features and 41 all-comment features after shared preprocessing: CardiffNLP, TextDetox, semantic, temporal, history, structural, and all 20 continuous AQuA expected ordinal predictors. The AQuA hard labels, both composites, and mean toxicity remain descriptive/sensitivity fields. Stage 9 will assemble publication outputs after the factorial winners are frozen from development CV.